In [34]:
import os
import random
from pathlib import Path

import numpy as np
import cv2
import tensorflow as tf
import sys
from google.colab import drive
drive.mount('/content/drive')
sys.path.append('/content/drive/MyDrive/HCI Project/Datasets')
from utils import generate_transition_linear

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [35]:
import os
import sys

# 1. This MUST happen before any other TF imports
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("✅ GPU Memory Growth enabled")
    except Exception as e:
        print(f"⚠️ Could not set memory growth: {e}")

# 2. Setup Drive and Paths
from google.colab import drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Add folder for utils.py (FOLDER path only)
sys.path.append('/content/drive/MyDrive/HCI Project/Datasets')

# 3. Imports
from utils import generate_transition_linear
import numpy as np

# 4. Load Model
MODEL_PATH = "/content/drive/MyDrive/HCI Project/models/bio_exergame_multimodal.keras"
if os.path.exists(MODEL_PATH):
    model = tf.keras.models.load_model(MODEL_PATH)
    print("✅ Model loaded successfully on T4 GPU!")
else:
    print(f"❌ Model file not found at: {MODEL_PATH}")


✅ GPU Memory Growth enabled
✅ Model loaded successfully on T4 GPU!


In [36]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

MODEL_PATH   = "/content/drive/MyDrive/HCI Project/models/bio_exergame_multimodal.keras"

SEGMENT_DIR  = "/content/drive/MyDrive/HCI Project/Datasets/ibdts_processed_data/ibdts_processed_data/motion"
RESULT_DIR   = "/content/drive/MyDrive/HCI Project/Results"
os.makedirs(RESULT_DIR, exist_ok=True)

TARGET_DURATION = 960

In [37]:
# weights for different cost components
W_LEN      = 0.8
W_SMOOTH   = 0.89
W_VARIETY  = 0.001
W_INT      = 0.8
W_BIO      = 1.0

# simulated annealing params
MAX_ITER        = 1000
INIT_TEMP       = 1.0
FINAL_TEMP      = 0.01
COOLING_RATE    = 0.99

# model input config
T      = 16
H, W   = 128, 128
HR_LEN = 4
HR_CLASS_NAMES   = ["low", "medium", "high"]
DIFF_CLASS_NAMES = ["low", "medium", "high"]

In [38]:
TARGET_HR_CLASS   = 1  # medium
TARGET_DIFF_CLASS = 1  # medium

In [39]:
import tensorflow as tf

# Configure GPU memory growth
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("✅ GPU Memory Growth enabled")
    except RuntimeError as e:
        print(f"❌ GPU Config Error: {e}")

# Now import your other libraries
import os
import sys
from google.colab import drive
drive.mount('/content/drive')
sys.path.append('/content/drive/MyDrive/HCI Project/Datasets')


✅ GPU Memory Growth enabled
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [40]:
model = tf.keras.models.load_model(MODEL_PATH)

In [41]:
def predict_segment_state(video_clip, hr_seq):
    """
    video_clip: (16,128,128,3) float32 in [0,1]
    hr_seq:     (4,) float32
    returns dict with hr_probs, diff_probs, perf
    """
    x = {
        "video_in": video_clip[None, ...],
        "hr_in":    hr_seq[None, ...],
    }
    hr_logits, diff_logits, perf = model.predict(x, verbose=0)

    return {
        "hr_probs":   hr_logits[0],
        "diff_probs": diff_logits[0],
        "perf":       float(perf[0, 0]),
    }

In [42]:
def load_segments(segment_dir, slice_len=120): # slice_len = 4 seconds @ 30fps
    files = [os.path.join(segment_dir, f) for f in sorted(os.listdir(segment_dir)) if f.endswith(".npy")]
    segments = []
    print(f"Processing {len(files)} files and slicing into {slice_len}-frame chunks...")

    for f in files:
        data = np.load(f, allow_pickle=True)
        # Skip files with errors
        if np.any(np.isnan(data)):
            continue

        # SLICING LOGIC: Loop through the frames in steps of slice_len
        # This turns 1 long file into ~8 shorter segments
        if data.shape != ():
             for start in range(0, len(data) - slice_len + 1, slice_len):
                chunk = data[start : start + slice_len]
                root  = chunk[:, 0, :]
                pred  = chunk - chunk[:, 0:1, :]

                segments.append({
                    "pred_position": pred,
                    "root_position": root,
                    "video_path": None,
                    "start_frame": start
                })
        else:
            segments.append(data.item())

    for i, seg in enumerate(segments):
        seg["seg_idx"] = i

    print(f"✅ Total segments created: {len(segments)}")
    return segments



def sample_clip_from_video(video_path, start_frame, end_frame, n_frames=T):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        # fallback: black clip
        cap.release()
        return np.zeros((n_frames, H, W, 3), dtype=np.float32)

    # sample indices evenly
    idxs = np.linspace(start_frame, max(start_frame, end_frame - 1),
                       n_frames, dtype=int)
    frames = []
    for idx in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, frame = cap.read()
        if not ok:
            frame = np.zeros((H, W, 3), dtype=np.uint8)
        else:
            frame = cv2.resize(frame, (W, H))
            frame = frame[..., ::-1]  # BGR -> RGB
        frames.append(frame.astype(np.float32) / 255.0)
    cap.release()
    return np.stack(frames, axis=0)

def synthetic_hr_seq_from_class(hr_class, hr_len=HR_LEN):
    # rough mapping for optimization (tune these values)
    SYNTH_HR_VALUES = {0: 80.0, 1: 115.0, 2: 150.0}
    base = SYNTH_HR_VALUES[int(hr_class)]
    return np.full((hr_len,), base, dtype=np.float32)

In [43]:
def initial_pose(available_segments):
    idx = np.random.randint(len(available_segments))
    return available_segments[idx]

def targetlen_cost(sequence, duration, w_len):
    current_len = sum(len(seg['pred_position']) for seg in sequence)
    return w_len * (1.0 / duration) * abs(current_len - duration)

def normalize_displacement(d, scale=0.1):
    return (np.tanh(d * scale) + 1.0) / 2.0

def smoothness_cost(sequence):
    T_trans = len(sequence) - 1
    if T_trans <= 0:
        return 0.0  # Change from inf to 0.0 for single segments

    total_disp = 0.0
    for i in range(T_trans):
        last_pose  = sequence[i]['pred_position'][-1]
        first_pose = sequence[i+1]['pred_position'][0]
        disp = np.abs(last_pose - first_pose)
        total_disp += np.sum(disp)

    avg_disp = total_disp / T_trans
    return normalize_displacement(avg_disp)

def total_cost(sequence, duration, w_len, w_smooth, w_var, w_int, w_bio, candidate_id=None, debug=False):
    existing_ids = [segment_id(seg) for seg in sequence]

    c_len    = targetlen_cost(sequence, duration, w_len)
    c_smooth = smoothness_cost(sequence) * w_smooth
    c_var    = variety_cost(candidate_id, existing_ids) * w_var
    c_int    = intensity_cost(sequence) * (1.0 - w_int)
    c_bio    = segment_bio_cost(sequence) * w_bio

    total = c_len + c_smooth + c_var + c_int + c_bio

    if debug or np.isinf(total) or np.isnan(total):
        print(f"DEBUG - len: {len(sequence)}, total: {total:.4f}, "
              f"len_c: {c_len:.4f}, smooth_c: {c_smooth:.4f}, "
              f"var_c: {c_var:.4f}, int_c: {c_int:.4f}, bio_c: {c_bio:.4f}")

    return total


def segment_id(seg):
    return np.mean(seg['pred_position'], axis=0)

def variety_cost(candidate_id, existing_ids):
    if candidate_id is None or not existing_ids:
        return 0.0
    dists = [np.linalg.norm(candidate_id - eid) for eid in existing_ids]
    return max(min(dists), 1e-2)

def intensity_cost(sequence):
    if len(sequence) == 0:
        return float('inf')
    total = 0.0
    M = np.ones(24)
    for seg in sequence:
        v   = np.diff(seg['pred_position'], axis=0)
        v2  = np.sum(v**2, axis=-1)
        ke  = 0.5 * np.sum(M * v2, axis=-1)
        total += np.sum(ke)
    return total

def segment_bio_cost(sequence):
    """
    For each segment, run the model on its (video, synthetic HR) window
    and penalize deviation from target classes + low predicted perf.
    (For speed, you may cache per-segment predictions.)
    """
    if len(sequence) == 0:
        return 0.0

    hr_devs, diff_devs, perf_vals = [], [], []

    for seg in sequence:
        video_path  = seg.get("video_path", None)
        start_frame = int(seg.get("start_frame", 0))
        end_frame   = int(seg.get("end_frame", start_frame + 120))  # ~4s @30fps
        hr_class    = int(seg.get("hr_class", TARGET_HR_CLASS))

        if video_path is None:
            continue

        video_clip = sample_clip_from_video(video_path, start_frame, end_frame)
        hr_seq     = synthetic_hr_seq_from_class(hr_class)

        pred = predict_segment_state(video_clip, hr_seq)

        hr_probs   = pred["hr_probs"]
        diff_probs = pred["diff_probs"]
        perf       = pred["perf"]

        hr_cls   = int(np.argmax(hr_probs))
        diff_cls = int(np.argmax(diff_probs))

        hr_devs.append(abs(hr_cls   - TARGET_HR_CLASS))
        diff_devs.append(abs(diff_cls - TARGET_DIFF_CLASS))
        perf_vals.append(perf)

    if len(hr_devs) == 0:
        return 0.0

    hr_cost   = float(np.mean(hr_devs))
    diff_cost = float(np.mean(diff_devs))
    perf_mean = float(np.mean(perf_vals))
    perf_cost = max(0.0, 1.0 - perf_mean)  # penalize perf < 1.0

    return hr_cost + diff_cost + perf_cost

def total_cost(sequence,
               duration,
               w_len,
               w_smooth,
               w_var,
               w_int,
               w_bio,
               candidate_id=None):
    existing_ids = [segment_id(seg) for seg in sequence]
    c_len   = targetlen_cost(sequence, duration, w_len)
    c_smooth= smoothness_cost(sequence) * w_smooth
    c_var   = variety_cost(candidate_id, existing_ids) * w_var
    c_int   = intensity_cost(sequence) * (1.0 - w_int)
    c_bio   = segment_bio_cost(sequence) * w_bio
    return c_len + c_smooth + c_var + c_int + c_bio

In [44]:
def optimization_step(sequence,
                      available_segments,
                      duration,
                      w_len,
                      w_smooth,
                      w_var,
                      w_int,
                      w_bio,
                      temperature):
    current_cost = total_cost(sequence, duration, w_len, w_smooth, w_var,
                              w_int, w_bio, candidate_id=None)
    current_len  = sum(len(seg['pred_position']) for seg in sequence)
    trial_seq    = sequence.copy()
    candidate_id = None

    if len(trial_seq) == 0:
        move_type      = "add"
        selected_index = 0
    else:
        selected_index = random.randrange(len(trial_seq))
        if current_len >= duration:
            move_types = ["remove", "modify"]
            move_probs = [0.3, 0.7]
        else:
            move_types = ["add", "remove", "modify"]
            move_probs = [0.4, 0.2, 0.4]
        move_type = np.random.choice(move_types, p=move_probs)

    if move_type == "add" and current_len < duration:
        seg_add = random.choice(available_segments)
        before_or_after = random.choice(["before", "after"])
        if before_or_after == "before":
            trial_seq.insert(selected_index, seg_add)
        else:
            trial_seq.insert(selected_index + 1, seg_add)
        candidate_id = segment_id(seg_add)

    elif move_type == "remove" and len(trial_seq) > 1:
        trial_seq.pop(selected_index)

    elif move_type == "modify":
        seg_new = random.choice(available_segments)
        trial_seq[selected_index] = seg_new
        candidate_id = segment_id(seg_new)

    trial_cost = total_cost(trial_seq, duration, w_len, w_smooth, w_var,
                            w_int, w_bio, candidate_id=candidate_id)

    delta = trial_cost - current_cost
    accept_prob = np.exp(-delta / temperature)

    if (trial_cost < current_cost) or (np.random.random() < accept_prob):
        return trial_seq, trial_cost
    else:
        return sequence, current_cost

In [45]:
def stitch_sequence(sequence, save_path):
    origin = np.zeros((1, 3))
    previous_end_root = origin

    final_pred = []
    final_root = []
    final_global = []

    for i, seg in enumerate(sequence):
        if i == 0:
            offset = origin - seg['root_position'][0]
        else:
            offset = previous_end_root - seg['root_position'][0]

        adj_root = seg['root_position'] + offset

        final_pred.extend(seg['pred_position'])
        final_root.extend(adj_root)

        previous_end_root = adj_root[-1]

        if i < len(sequence) - 1:
            next_seg = sequence[i+1]
            trans_pred = generate_transition_linear(
                seg['pred_position'][-1],
                next_seg['pred_position'][0],
                5
            )
            trans_root = generate_transition_linear(
                adj_root[-1],
                next_seg['root_position'][0] +
                (previous_end_root - next_seg['root_position'][0]),
                5
            )
            final_pred.extend(trans_pred)
            final_root.extend(trans_root)

    final_pred  = np.array(final_pred)
    final_root  = np.array(final_root)

    for pred, root in zip(final_pred, final_root):
        final_global.append(pred + root)
    final_global = np.array(final_global)

    data = {
        "pred_position": final_global,
        "root_position": final_root
    }
    np.save(save_path, data)

In [50]:
if __name__ == "__main__":
    available_segments = load_segments(SEGMENT_DIR)
    print(f"Loaded {len(available_segments)} segments.")

    current_sequence = [initial_pose(available_segments)]
    current_cost = total_cost(current_sequence,
                              TARGET_DURATION,
                              W_LEN,
                              W_SMOOTH,
                              W_VARIETY,
                              W_INT,
                              W_BIO,
                              candidate_id=None)

    temp      = INIT_TEMP
    iteration = 0

    while temp > FINAL_TEMP and iteration < MAX_ITER:
        print(f"Iter {iteration}, temp {temp:.4f}, "
              f"len {len(current_sequence)}, cost {current_cost:.4f}")
        current_sequence, current_cost = optimization_step(
            current_sequence,
            available_segments,
            TARGET_DURATION,
            W_LEN,
            W_SMOOTH,
            W_VARIETY,
            W_INT,
            W_BIO,
            temp
        )
        temp *= COOLING_RATE
        iteration += 1

    out_path = os.path.join(RESULT_DIR, "stitched_motion.npy")
    stitch_sequence(current_sequence, out_path)
    print("Saved stitched motion to:", out_path)

Processing 80 files and slicing into 120-frame chunks...
✅ Total segments created: 640
Loaded 640 segments.
Iter 0, temp 1.0000, len 1, cost 0.9055
Iter 1, temp 0.9900, len 1, cost 0.9541
Iter 2, temp 0.9801, len 2, cost 1.5865
Iter 3, temp 0.9703, len 3, cost 1.5441
Iter 4, temp 0.9606, len 3, cost 1.5441
Iter 5, temp 0.9510, len 4, cost 1.5111
Iter 6, temp 0.9415, len 5, cost 1.5674
Iter 7, temp 0.9321, len 5, cost 1.8172
Iter 8, temp 0.9227, len 5, cost 1.8172
Iter 9, temp 0.9135, len 5, cost 1.8201
Iter 10, temp 0.9044, len 5, cost 1.6518
Iter 11, temp 0.8953, len 5, cost 1.6264
Iter 12, temp 0.8864, len 6, cost 1.6282
Iter 13, temp 0.8775, len 7, cost 1.6124
Iter 14, temp 0.8687, len 8, cost 1.6793
Iter 15, temp 0.8601, len 8, cost 1.6048
Iter 16, temp 0.8515, len 7, cost 1.6125
Iter 17, temp 0.8429, len 8, cost 1.5849
Iter 18, temp 0.8345, len 7, cost 1.6125
Iter 19, temp 0.8262, len 8, cost 1.5987
Iter 20, temp 0.8179, len 8, cost 1.7171
Iter 21, temp 0.8097, len 8, cost 1.6181


In [47]:
import os

# Let's verify the contents of the segment directory
segment_dir = "/content/drive/MyDrive/HCI Project/Datasets/ibdts_processed_data"

if os.path.exists(segment_dir):
    print(f"Directory exists: {segment_dir}")
    files = os.listdir(segment_dir)
    print(f"Found {len(files)} files.")
    if len(files) > 0:
        print("First 5 files:", files[:5])
    else:
        print("Directory is empty!")
else:
    print(f"Directory does NOT exist: {segment_dir}")

Directory exists: /content/drive/MyDrive/HCI Project/Datasets/ibdts_processed_data
Found 1 files.
First 5 files: ['ibdts_processed_data']


In [48]:
import numpy as np
import os

search_dir = "/content/drive/MyDrive/HCI Project/Datasets/ibdts_processed_data/ibdts_processed_data/motion"
print(f"Checking files in: {search_dir}")

files = sorted([f for f in os.listdir(search_dir) if f.endswith('.npy')])

if not files:
    print("No .npy files found.")
else:
    print(f"Found {len(files)} files. Checking the first few:\n")

    for i, fname in enumerate(files[:5]):
        path = os.path.join(search_dir, fname)
        try:
            data = np.load(path, allow_pickle=True)
            print(f"[{i}] File: {fname}")
            print(f"    Type: {type(data)}")
            print(f"    Shape: {data.shape}")

            # Try to peek at contents
            if data.ndim == 0:
                print("    Content is 0-d array (scalar). wrapped type:", type(data.item()))
                print("    Keys:", data.item().keys() if isinstance(data.item(), dict) else "Not a dict")
            else:
                print("    Content is N-d array.")
        except Exception as e:
            print(f"[{i}] File: {fname} - Error loading: {e}")
        print("-" * 30)

Checking files in: /content/drive/MyDrive/HCI Project/Datasets/ibdts_processed_data/ibdts_processed_data/motion
Found 80 files. Checking the first few:

[0] File: p10_ie.npy
    Type: <class 'numpy.ndarray'>
    Shape: (1020, 24, 3)
    Content is N-d array.
------------------------------
[1] File: p10_ih.npy
    Type: <class 'numpy.ndarray'>
    Shape: (1020, 24, 3)
    Content is N-d array.
------------------------------
[2] File: p10_le.npy
    Type: <class 'numpy.ndarray'>
    Shape: (1020, 24, 3)
    Content is N-d array.
------------------------------
[3] File: p10_lh.npy
    Type: <class 'numpy.ndarray'>
    Shape: (1020, 24, 3)
    Content is N-d array.
------------------------------
[4] File: p11_ie.npy
    Type: <class 'numpy.ndarray'>
    Shape: (1020, 24, 3)
    Content is N-d array.
------------------------------


In [49]:
def load_segments(segment_dir, slice_len=120):
    files = [os.path.join(segment_dir, f) for f in sorted(os.listdir(segment_dir)) if f.endswith(".npy")]
    segments = []
    print(f"Processing {len(files)} files and slicing into {slice_len}-frame chunks...")

    for f in files:
        data = np.load(f, allow_pickle=True)

        # Log which files are skipped due to NaNs
        if np.any(np.isnan(data)):
            print(f"⚠️ Skipping {os.path.basename(f)} (contains NaNs)")
            continue

        if data.shape != ():
             for start in range(0, len(data) - slice_len + 1, slice_len):
                chunk = data[start : start + slice_len]
                root  = chunk[:, 0, :]
                pred  = chunk - chunk[:, 0:1, :]

                segments.append({
                    "pred_position": pred,
                    "root_position": root,
                    "video_path": None,
                    "start_frame": start
                })
        else:
            segments.append(data.item())

    for i, seg in enumerate(segments):
        seg["seg_idx"] = i

    print(f"✅ Total segments created: {len(segments)}")
    return segments
